In [0]:
from pyspark.sql import functions as F

bronze_geolocation = spark.table("brazilian_ecommerce.prod.bronze_geolocation")

staging_geolocation = (
    bronze_geolocation
    # zip code prefix stays as string to preserve leading zeros
    .withColumn("geolocation_lat", F.col("geolocation_lat").cast("double"))
    .withColumn("geolocation_lng", F.col("geolocation_lng").cast("double"))
    .withColumn("geolocation_city", F.trim(F.lower(F.col("geolocation_city"))))
    .withColumn("geolocation_state", F.trim(F.upper(F.col("geolocation_state"))))
    .filter(F.col("geolocation_zip_code_prefix").isNotNull())
    .dropDuplicates()   # drop only exact full-row duplicates, not repeated zip prefixes
)

full_table_name = "brazilian_ecommerce.prod.staging_geolocation"

(
    staging_geolocation.write
    .mode("overwrite")
    .option("comment", "geolocation staging table")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

print(f"Loaded {staging_geolocation.count()} rows into {full_table_name}")